# Phase 1: Exploratory Data Analysis (EDA) on GSM8K

This notebook analyzes the token length distributions, reasoning step counts, and answer characteristics of the frozen GSM8K splits using the `deepseek-ai/deepseek-math-7b-base` tokenizer.

### Objectives
1. **Token Length Distributions:** Quantify question, solution, and prompt+solution lengths under the DeepSeek tokenizer to establish optimal sequence length limits (`max_seq_length`, `max_new_tokens`).
2. **Step Count Distribution:** Measure the number of intermediate reasoning steps (lines before `####`).
3. **Answer Types:** Verify answer composition (integer, decimal, negative) and formatting characteristics across all 8,792 problems.
4. **Few-Shot Prompt Budgeting:** Profile the 8-shot chain-of-thought prompt length for baseline evaluation.

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Ensure project root is accessible
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
from transformers import AutoTokenizer
from src.data.formatting import format_zero_shot_prompt, format_few_shot_prompt
from src.math_utils.normalization import parse_numeric_value

## 2. Load Processed Dataset Splits
Loads the frozen train, validation (analysis + gating), and test splits created in Day 7.

In [ ]:
data_dir = PROJECT_ROOT / "data" / "processed"
splits = {
    "train": pd.read_parquet(data_dir / "train" / "train.parquet"),
    "val": pd.read_parquet(data_dir / "val" / "val.parquet"),
    "val_analysis": pd.read_parquet(data_dir / "val" / "val_analysis.parquet"),
    "val_gating": pd.read_parquet(data_dir / "val" / "val_gating.parquet"),
    "test": pd.read_parquet(data_dir / "test" / "test.parquet"),
}

for name, df in splits.items():
    print(f"{name:<15}: {len(df):>5} problems (columns: {list(df.columns)})")

## 3. Load DeepSeek Math Tokenizer
Loads `deepseek-ai/deepseek-math-7b-base` (vocab size: 100,000).

In [ ]:
model_name = "deepseek-ai/deepseek-math-7b-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"Loaded tokenizer: {model_name}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

## 4. Load or Run Precomputed EDA Summary
The headless script `scripts/run_eda.py` computes full tokenization over all 8,792 examples and writes `reports/eda_summary.json`.

In [ ]:
eda_file = PROJECT_ROOT / "reports" / "eda_summary.json"
if not eda_file.exists():
    from scripts.run_eda import run_eda
    summary = run_eda(output_dir=str(PROJECT_ROOT / "reports"))
else:
    with open(eda_file, "r", encoding="utf-8") as f:
        summary = json.load(f)

print(f"Dataset: {summary['dataset']}")
print(f"Tokenizer: {summary['tokenizer']}")
print(f"Counts: {summary['counts']}")

## 5. Token Length Distributions
Reviewing token length percentiles across Questions, Solutions, and Combined inputs.

In [ ]:
tok_stats = summary["token_lengths"]["all"]

df_tokens = pd.DataFrame([
    {"Sequence": "Question", **tok_stats["question_tokens"]},
    {"Sequence": "Solution", **tok_stats["solution_tokens"]},
    {"Sequence": "Prompt + Solution", **tok_stats["combined_tokens"]},
])[["Sequence", "min", "p50", "p90", "p95", "p99", "max"]]

df_tokens.rename(columns={"p50": "median", "p90": "90th %", "p95": "95th %", "p99": "99th %"}, inplace=True)
display(df_tokens)

## 6. Reasoning Step Counts
Number of reasoning lines in solutions prior to the `####` final answer line.

In [ ]:
step_stats = summary["reasoning_steps"]["all"]
print(f"Reasoning Steps per Problem:")
print(f"  Min:    {step_stats['min']}")
print(f"  Mean:   {step_stats['mean']}")
print(f"  Median: {step_stats['p50']}")
print(f"  95th %: {step_stats['p95']}")
print(f"  Max:    {step_stats['max']}")

## 7. Answer Composition Analysis
Validates integer vs. decimal vs. negative answer distributions across the full corpus.

In [ ]:
ans = summary["answer_types"]["all"]
print(f"Total Parsed: {ans['total_parsed']} / 8792 ({ans['total_parsed']/8792*100:.1f}%)")
print(f"Integers:     {ans['integer_count']} ({ans['integer_pct']}%)")
print(f"Decimals:     {ans['decimal_count']} ({ans['decimal_pct']}%)")
print(f"Negatives:    {ans['negative_count']} ({ans['negative_pct']}%)")

## 8. 8-Shot Few-Shot Prompt Length
Evaluates the context window requirements for the fixed 8-shot few-shot CoT evaluation prompt.

In [ ]:
fs = summary["few_shot_prompt_tokens"]
print(f"8-Shot Few-Shot Prompt Tokens:")
print(f"  Min:     {fs['min']}")
print(f"  Median:  {fs['p50']}")
print(f"  95th %:  {fs['p95']}")
print(f"  Max:     {fs['max']}")
print(f"Total context required (prompt + 512 new tokens): ~{int(fs['max'] + 512)} tokens (fits comfortably within 2048/4096 context windows).")

## 9. Sequence Budget Recommendations & Config Updates

Based on the empirical token distributions:
- **`max_new_tokens = 512`**: 99% of GSM8K solutions fit within 221 tokens (max observed is 343). 512 tokens provides more than enough headroom even for verbose reasoning.
- **`max_seq_length = 512`**: 99% of zero-shot prompt+solution pairs fit within 337 tokens (max observed is 465). 512 tokens accommodates 100% of zero-shot training pairs without truncation while keeping memory footprint lean on Colab GPU runtimes.